In [50]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential,layers
import tensorflow_datasets as tfds

In [51]:
(ds_train,ds_test),ds_info =tfds.load(
    'imdb_reviews',
    split=['train','train'],
    as_supervised=True,
    with_info=True
)

In [29]:
for text,label in ds_train.take(2):
  print(text,label)

tf.Tensor(b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.", shape=(), dtype=string) tf.Tensor(0, shape=(), dtype=int64)
tf.Tensor(b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on t

In [49]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
import re
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

stopwords_list = stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [31]:
def preprocessing(text,label):
  text = text.numpy().decode('utf-8')
  text = text.lower()
  text = re.sub('r[^a-z]\s]','',text)
  tokenized_text =  word_tokenize(text)
  preprocessed_text = ''.join([WordNetLemmatizer().lemmatize(word) for word in tokenized_text if word not in stopwords_list])
  return preprocessed_text,label

<>:4: SyntaxWarning: invalid escape sequence '\s'
<>:4: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-1754790487.py:4: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('r[^a-z]\s]','',text)


In [32]:
def preprocessing_tf(text,label):
  text,label =tf.py_function(preprocessing,inp=[text,label],Tout=[tf.string,label.dtype])
  text.set_shape([])
  label.set_shape([])
  return text,label

In [33]:
ds_train = ds_train.map(preprocessing_tf)
ds_test = ds_test.map(preprocessing_tf)

In [34]:
for text,label in ds_train.take(2):
  print(text,label)

tf.Tensor(b"absolutelyterriblemovie.n'tluredchristopherwalkenmichaelironside.greatactor,mustsimplyworstrolehistory.evengreatactingcouldredeemmovie'sridiculousstoryline.movieearlyninetyupropagandapiece.patheticscenecolumbianrebelmakingcaserevolution.mariaconchitaalonsoappearedphony,pseudo-loveaffairwalkennothingpatheticemotionalplugmoviedevoidrealmeaning.disappointedmovielike,ruiningactor'slikechristopherwalken'sgoodname.couldbarelysit.", shape=(), dtype=string) tf.Tensor(0, shape=(), dtype=int64)
tf.Tensor(b'knownfallasleepfilm,usuallyduecombinationthingincluding,reallytired,warmcomfortablesetteeatenlot.howeveroccasionfellasleepfilmrubbish.plotdevelopmentconstant.constantlyslowboring.thingseemedhappen,explanationcausing.admit,maymissedpartfilm,watchedmajorityeverythingseemedhappenaccordwithoutrealconcernanythingelse.cantrecommendfilm.', shape=(), dtype=string) tf.Tensor(0, shape=(), dtype=int64)


In [37]:
from tensorflow.keras.preprocessing.text import Tokenizer
tokenizer = Tokenizer(num_words=10000, oov_token = 'UNK')
train_texts = [text.numpy().decode('utf-8') for text, label in ds_train]
tokenizer.fit_on_texts(train_texts)

In [38]:
def encode(text, label):
  seuqeunces = tokenizer.texts_to_sequences(text)
  padded_sequences = tensorflow.keras.sequences.pad_sequences(sequences, maxlen=256)
  return padded_sequences

In [39]:
def encode_tf(text, label):
  text, label = tf.py_function(encode, inp = [text, label], Tout = [tf.int64, label.dtype])
  text.set_shape([None])
  label.set_shape([])
  return text, label

In [40]:
ds_train = ds_train.map(encode_tf)
ds_test = ds_test.map(encode_tf)

In [52]:
vectorizer = layers.TextVectorization(max_tokens = 10000, output_sequence_length = 256)
vectorizer.adapt(ds_train.map(lambda text, label: text))
ds_train = ds_train.map(lambda text, label:(vectorizer(text), label))
ds_test = ds_test.map(lambda text, label:(vectorizer(text), label))

In [53]:
for text, label in ds_train.take(2):
  print(text, label)

tf.Tensor(
[  11   14   34  412  384   18   90   28    1    8   33 1322 3560   42
  487    1  191   24   85  152   19   11  217  316   28   65  240  214
    8  489   54   65   85  112   96   22 5596   11   93  642  743   11
   18    7   34  394 9522  170 2464  408    2   88 1216  137   66  144
   51    2    1 7558   66  245   65 2870   16    1 2860    1    1 1426
 5050    3   40    1 1579   17 3560   14  158   19    4 1216  891 8040
    8    4   18   12   14 4059    5   99  146 1241   10  237  704   12
   48   24   93   39   11 7339  152   39 1322    1   50  398   10   96
 1155  851  141    9    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0 

In [54]:
ds_train = ds_train.batch(64)
ds_test = ds_test.batch(64)

In [55]:
model = Sequential(
    [
        layers.Embedding(10000, 512),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation = 'relu'),
        layers.Dense(1, activation = 'sigmoid')
    ]
)

In [59]:
optimizer = keras.optimizers.Adam()
loss_fn = keras.losses.BinaryCrossentropy()
acc_metric = keras.metrics.BinaryAccuracy()

In [ ]:
epochs = 10
for i in range(epochs):
  loss = 0
  print(f"Training started for epoch number {i+1}")
  for batch_id, (x_batch, y_batch) in enumerate(ds_train):
    with tf.GradientTape() as tape:
      y_pred = model(x_batch, training = True)
      batch_loss = loss_fn(y_batch, y_pred)
    gradients = tape.gradient(batch_loss, model.trainable_weights)
    optimizer.apply_gradients(zip(gradients, model.trainable_weights))
    acc_metric.update_state(y_batch, y_pred)
  loss += batch_loss
  accuracy = acc_metric.result()
  print(f'training accuracy over epoch is {accuracy} and the loss is {loss} ')
  acc_metric.reset_state()

Training started for epoch number 1
training accuracy over epoch is 0.7701200246810913 and the loss is 0.39799726009368896 
Training started for epoch number 2


In [ ]:
for (x_batch, y_batch) in ds_test:
  y_pred = model(x_batch, training = False)
  acc_metric.update_state(y_batch, y_pred)
accuracy = acc_metric.result()
print(f"Test accuracy is {accuracy}")